In [1]:
# load data for bagpipes from txt file
import pandas as pd
import numpy as np

df = pd.read_csv("Table_for_bagpipes.txt", sep="\t", header=None, index_col=0, na_values="NaN")
SIDs = np.arange(25300,25330)
GN = []
for i in SIDs:
    gal_num = int(df.iloc[i, 0])
    GN.append(gal_num)

In [3]:
# run catalog SED fitting for matching sources
# photometry load function

import matplotlib.pyplot as plt
import numpy as np 
import bagpipes as pipes
from astropy.io import fits

def load_goodss(ID):
    """ Load photometry from flux catalogue. """

    # load up the relevant columns from the catalogue.
    cat = np.loadtxt("Table_for_bagpipes.txt",
                     usecols=(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18))
    
    # Find the correct row for the object we want.
    row = int(ID) - 1

    # Extract the object we want from the catalogue.

    fluxes =  cat[row, 4:11]
    fluxerrs =  cat[row, 11:18]

    # Turn these into a 2D array.
    photometry = np.c_[fluxes, fluxerrs]
    

    # blow up the errors associated with any missing fluxes.
    for i in range(len(photometry)):
        if np.isnan(photometry[i, 0]):
            photometry[i,:] = [0., 9.9*10**99.]
        if (photometry[i, 0] <= 0):
            photometry[i,:] = [0., 3 * photometry[i, 1] ]

    # Enforce a maximum SNR of 20, or 10 in the Infra-Red Array Camera channels. 
    for i in range(len(photometry)):
        if i < 2:                       # here for blue
            max_snr = 20.               # max fluxerr = 1/20 flux
            
        else:                           # here for green and red
            max_snr = 10.               # max fluxerr = 1/10 flux
        
        if photometry[i, 0]/photometry[i, 1] > max_snr:
            photometry[i, 1] = photometry[i, 0]/max_snr

    return photometry

In [5]:
# Let's pick object 1441 as a test case to check this works.
print(load_goodss("29200"))

[[ 97.42961758  23.03355457]
 [213.70478854  19.43532347]
 [320.20567732  32.02056773]
 [335.50158413  39.68491898]
 [349.26674158  34.92667416]
 [282.94024037  28.29402404]
 [201.57722232  22.63233729]]


In [7]:
# load filter list
SED_filt_list = ["filters/HST_ACS_WFC.F606W.txt",
                 "filters/HST_ACS_WFC.F814W.txt",
                 "filters/HST_WFC3_IR.F105W.txt",
                 "filters/HST_WFC3_IR.F140W.txt",
                 "filters/JWST_NIRCam.F277W.txt",
                 "filters/JWST_NIRCam.F356W.txt",
                 "filters/JWST_NIRCam.F444W.txt"]

In [9]:
# load catalog fitting instructions
import numpy as np 
import bagpipes as pipes
from astropy.io import fits

exp = {}                                  # Tau-model star-formation history component
exp["age"] = (0.1, 15.)                   # Vary age between 100 Myr and 15 Gyr. In practice 
                                          # the code automatically limits this to the age of
                                          # the Universe at the observed redshift.

exp["tau"] = (0.3, 10.)                   # Vary tau between 300 Myr and 10 Gyr
exp["massformed"] = (1., 15.)             # vary log_10(M*/M_solar) between 1 and 15
exp["metallicity"] = (0., 2.5)            # vary Z between 0 and 2.5 Z_oldsolar

dust = {}                                 # Dust component
dust["type"] = "Calzetti"                 # Define the shape of the attenuation curve
dust["Av"] = (0., 2.)                     # Vary Av between 0 and 2 magnitudes

fit_instructions = {}                     # The fit instructions dictionary
fit_instructions["redshift"] = (0.307, 0.309)  # Vary observed redshift from 0.99 to 1.01
fit_instructions["exponential"] = exp   
fit_instructions["dust"] = dust

In [11]:
# run catalog SED fitting with sources IDs (=matching sources)

fit_cat = pipes.fit_catalogue(SIDs, fit_instructions, load_goodss, spectrum_exists=False,
                              cat_filt_list=SED_filt_list, run="catalogue")

fit_cat.fit(verbose=False, sampler='multinest')

In [13]:
# transfer h5 files from .../posterior/catalogue to .../posterior

In [15]:
# generate the .png files for all sources
# flux data
for i in range(len(SIDs)):
    galaxy = pipes.galaxy(SIDs[i], load_goodss, spectrum_exists=False, filt_list=SED_filt_list)
    fig = galaxy.plot(show=False)[0]
    num = SIDs[i]
    name = f"pipes/plots/{num}_flux.png"
    fig.savefig(name, format="png", bbox_inches="tight")
    plt.close(fig)


In [17]:
# generate the .png files for all sources 
# SED fitting
import matplotlib as mpl
plt.close()
for i in range(len(SIDs)):
    galaxy = pipes.galaxy(SIDs[i], load_goodss, spectrum_exists=False, filt_list=SED_filt_list)
    fit = pipes.fit(galaxy, fit_instructions)
    fit.fit(verbose=False)
    fit.posterior.get_advanced_quantities()
    fig = plt.figure(figsize=(12, 7))
    gs = mpl.gridspec.GridSpec(7, 4, hspace=3., wspace=0.1)
    ax1 = plt.subplot(gs[:4, :])
    pipes.plotting.add_observed_photometry(fit.galaxy, ax1, zorder=10)
    pipes.plotting.add_photometry_posterior(fit, ax1)
    labels = ["sfr", "mass_weighted_age", "stellar_mass", "ssfr"]
    post_quantities = dict(zip(labels, [fit.posterior.samples[l] for l in labels]))
    axes = []
    for j in range(4):
        axes.append(plt.subplot(gs[4:, j]))
        pipes.plotting.hist1d(post_quantities[labels[j]], axes[-1], smooth=True, label=labels[j])
    num = SIDs[i]
    name = f"pipes/plots/{num}_sed.png"
    plt.savefig(name, format="png", bbox_inches="tight") 
    plt.close(fig)


Results loaded from pipes/posterior/./25300.h5

Fitting not performed as results have already been loaded from pipes/posterior/./25300.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/./25301.h5

Fitting not performed as results have already been loaded from pipes/posterior/./25301.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/./25302.h5

Fitting not performed as results have already been loaded from pipes/posterior/./25302.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/./25303.h5

Fitting not performed as results have already been loaded from pipes/posterior/./25303.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/./25304.h5

Fitting not performed as results have already been loaded from pipes/posterior/./25304.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/./25305.h5

Fitting not perf

In [19]:
# generate the .png files for all sources
# corner plot 
for i in range(len(SIDs)):
    galaxy = pipes.galaxy(SIDs[i], load_goodss, spectrum_exists=False, filt_list=SED_filt_list)
    fit = pipes.fit(galaxy, fit_instructions)
    fit.fit(verbose=False, sampler='multinest')
    fig = fit.plot_corner(save=False, show=False)
    num = SIDs[i]
    name = f"pipes/plots/{num}_corner.png"
    fig.savefig(name, format="png", bbox_inches="tight")
    plt.close(fig)


Results loaded from pipes/posterior/./25300.h5

Fitting not performed as results have already been loaded from pipes/posterior/./25300.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/./25301.h5

Fitting not performed as results have already been loaded from pipes/posterior/./25301.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/./25302.h5

Fitting not performed as results have already been loaded from pipes/posterior/./25302.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/./25303.h5

Fitting not performed as results have already been loaded from pipes/posterior/./25303.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/./25304.h5

Fitting not performed as results have already been loaded from pipes/posterior/./25304.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/./25305.h5

Fitting not perf

In [21]:
# This cell produces a multi-page A4 PDF, one galaxy per page.
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from photutils.segmentation import SegmentationImage
import matplotlib.image as mpimg
from PIL import Image

with PdfPages("SED_report.pdf") as pdf:
    for i, sid in enumerate(SIDs):        
        j = GN[i]
        # create .pdf report with SED fitting results
        name1 = f"pipes/plots/{sid}_flux.png"
        name2 = f"pipes/plots/{sid}_sed.png"
        name3 = f"pipes/plots/{sid}_corner.png"
        img1 = mpimg.imread(name1)
        img2 = mpimg.imread(name2)
        img3 = mpimg.imread(name3)
        title1 = f"Galaxy {j}: flux data"
        title2 = f"Galaxy {j}: SED fitting"
        title3 = f"Galaxy {j}: corner plot"
        
        # A4 portrait size in inches
        A4_WIDTH = 8.27
        A4_HEIGHT = 11.69

        # Create page 1 with 2 rows, 1 column
        fig1 = plt.figure(figsize=(A4_WIDTH, A4_HEIGHT))
        gs = fig1.add_gridspec(2, 1)

        # Load and display first image
        ax1 = fig1.add_subplot(gs[0, 0])
        ax1.set_title(title1)
        ax1.imshow(img1)
        ax1.axis("off")

        # Load and display second image
        ax2 = fig1.add_subplot(gs[1, 0])
        ax2.set_title(title2)
        ax2.imshow(img2)
        ax2.axis("off")

        # Reduce white space
        plt.tight_layout()
        pdf.savefig(fig1)
        plt.close(fig1)

        # Create page 2 with 1 row, 1 column
        fig2 = plt.figure(figsize=(A4_WIDTH, A4_HEIGHT))
        gs = fig2.add_gridspec(1, 1)
        ax = fig2.add_subplot(gs[0, 0])
    
        # Load and display image
        ax.set_title(title3)
        ax.imshow(img3)
        ax.axis("off")

        # Reduce white space
        plt.tight_layout()
        pdf.savefig(fig2)
        plt.close(fig2)
        